# Modeling: propensity scoring with calibrated classifiers

This notebook trains Logistic Regression, Random Forest, and XGBoost for
upsell propensity scoring. Each model is wrapped with CalibratedClassifierCV
to produce well-calibrated probability estimates. We compare AUC, Brier score,
and calibration curves.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import sys
sys.path.insert(0, '..')

from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    roc_auc_score, brier_score_loss, log_loss,
    average_precision_score, classification_report
)

from src.data_loader import prepare_full_pipeline, get_feature_columns
from src.model import build_models, train_and_calibrate, evaluate_model

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

## 1. Stratified train/test split

In [ ]:
data = prepare_full_pipeline(path='../data/marketing_campaign.csv')

X_train = data['X_train']
X_test = data['X_test']
y_train = data['y_train']
y_test = data['y_test']
feature_names = data['feature_names']

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set:     {X_test.shape[0]} samples')
print(f'Features:     {X_train.shape[1]}')
print(f'Response rate (train): {y_train.mean():.3f}')
print(f'Response rate (test):  {y_test.mean():.3f}')

## 2. Build and train models

We train three base models and wrap each with isotonic calibration
using `CalibratedClassifierCV`. This is critical for propensity scoring
because we need the predicted probabilities to reflect true response rates.

In [ ]:
raw_models = build_models(seed=42)
print(f'Models to train: {list(raw_models.keys())}')

calibrated_models = train_and_calibrate(raw_models, X_train, y_train, seed=42)

## 3. Evaluate each model

In [ ]:
eval_results = {}
for name, model in calibrated_models.items():
    r = evaluate_model(name, model, X_test, y_test)
    eval_results[name] = r

In [ ]:
# Comparison table
comp = pd.DataFrame({
    name: {'AUC-ROC': r['auc'], 'Avg Precision': r['avg_precision'],
           'Brier Score': r['brier'], 'Log Loss': r['log_loss']}
    for name, r in eval_results.items()
}).T

comp = comp.sort_values('AUC-ROC', ascending=False)
print('Model comparison:')
comp.round(4)

## 4. AUC and Brier score comparison

- **AUC-ROC** measures discrimination: how well the model ranks responders
  above non-responders
- **Brier score** measures calibration: how close predicted probabilities
  are to actual outcomes (lower is better)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
model_names = list(comp.index)
colors = ['#2196F3', '#4CAF50', '#FF9800']

# AUC-ROC bars
axes[0].bar(model_names, comp['AUC-ROC'], color=colors[:len(model_names)])
axes[0].set_ylabel('AUC-ROC')
axes[0].set_title('Discrimination (higher is better)')
axes[0].set_ylim(0.5, 1.0)
for i, v in enumerate(comp['AUC-ROC']):
    axes[0].text(i, v, f'{v:.3f}', ha='center', va='bottom', fontsize=10)

# Brier score bars
axes[1].bar(model_names, comp['Brier Score'], color=colors[:len(model_names)])
axes[1].set_ylabel('Brier score')
axes[1].set_title('Calibration (lower is better)')
for i, v in enumerate(comp['Brier Score']):
    axes[1].text(i, v, f'{v:.4f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

## 5. Calibration curves

A well-calibrated model produces predicted probabilities that match
observed frequencies. For example, among customers assigned a 30%
probability, approximately 30% should actually respond.

In [ ]:
from src.model import plot_calibration

fig, ax = plt.subplots(figsize=(8, 6))
colors_map = {'Logistic Regression': '#2196F3', 'Random Forest': '#4CAF50', 'XGBoost': '#FF9800'}

for name, model in calibrated_models.items():
    y_prob = model.predict_proba(X_test)[:, 1]
    fraction_pos, mean_predicted = calibration_curve(y_test, y_prob, n_bins=10)
    ax.plot(mean_predicted, fraction_pos, 's-',
            label=name, color=colors_map.get(name, 'gray'), linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfectly calibrated')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Fraction of positives')
ax.set_title('Calibration curves (after isotonic calibration)')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Compare calibrated vs uncalibrated predictions

To verify that CalibratedClassifierCV improves probability estimates,
we compare Brier scores before and after calibration.

In [ ]:
print(f'{"Model":25s}  {"Uncalibrated Brier":>18s}  {"Calibrated Brier":>16s}  {"Improvement":>11s}')
print('-' * 75)

for name, raw_model in raw_models.items():
    # Uncalibrated
    y_prob_raw = raw_model.predict_proba(X_test)[:, 1]
    brier_raw = brier_score_loss(y_test, y_prob_raw)

    # Calibrated
    y_prob_cal = calibrated_models[name].predict_proba(X_test)[:, 1]
    brier_cal = brier_score_loss(y_test, y_prob_cal)

    improvement = (brier_raw - brier_cal) / brier_raw * 100
    print(f'{name:25s}  {brier_raw:18.4f}  {brier_cal:16.4f}  {improvement:+10.1f}%')

## 7. ROC curves

In [ ]:
from sklearn.metrics import roc_curve

fig, ax = plt.subplots(figsize=(8, 6))

for name, r in eval_results.items():
    y_prob = r['y_prob']
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    ax.plot(fpr, tpr, label=f'{name} (AUC={r["auc"]:.3f})',
            color=colors_map.get(name, 'gray'), linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax.set_xlabel('False positive rate')
ax.set_ylabel('True positive rate')
ax.set_title('ROC curves - propensity models')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Select best model

In [ ]:
best_name = max(eval_results, key=lambda n: eval_results[n]['auc'])
best_auc = eval_results[best_name]['auc']
best_brier = eval_results[best_name]['brier']

print(f'Best model: {best_name}')
print(f'  AUC-ROC:     {best_auc:.4f}')
print(f'  Brier score: {best_brier:.4f}')
print(f'  Log loss:    {eval_results[best_name]["log_loss"]:.4f}')
print(f'\nThis model will be used for decile analysis and campaign targeting in 04_evaluation.ipynb')

## Summary

Three models were trained and calibrated for propensity scoring:

- **CalibratedClassifierCV** (isotonic method) improves probability estimates
  for all models, which is essential for decile-based targeting
- **Calibration curves** confirm that the wrapped models produce probabilities
  that closely track actual response rates
- **Brier score** measures calibration quality, complementing AUC which only
  measures ranking ability

Next: decile analysis, lift charts, and campaign ROI in `04_evaluation.ipynb`.